# **Ejercicio 2 - Desarrollo del sistema de descarga**

## **2.1 Análisis del script proporcionado**

Al analizar `scripts/download_data.py` se identificaron los siguientes puntos a modificar:

- **Año fijo en el código:** el script define `ANIO = 2026` como constante, por lo que no es posible descargar otro año sin editar el código, lo cual impide incorporar 2024 y 2025 más adelante.
- **Rutas dependientes del directorio de ejecución:** el destino se define como `Path("data/raw")`, que se resuelve a partir del directorio desde donde se ejecuta el script. Por ejemplo, si se ejecuta desde `notebooks/`, los datos terminan en `notebooks/data/raw/` y no en la estructura del proyecto.
- **Manejo de errores genérico:** cualquier respuesta distinta de éxito, incluyendo una falla de red, se reporta como "aún no publicado". No se logra distinguir si el mes realmente no existe o si hubo un problema de red o del servidor, de tal forma que el script puede terminar como exitoso sin haber descargado nada.
- **No se verifica que la descarga esté completa:** solo se valida que el archivo pese más de 0 bytes, aunque el servidor informa el tamaño esperado de cada archivo.
- **Sin espera entre intentos:** los reintentos se hacen de forma inmediata, uno tras otro. Si la fuente cuenta con algún tipo de rate limit, esto podría terminar en un bloqueo de nuestras peticiones.

## **2.2 Modificación del script**

Se aplicaron las modificaciones al script; dirigirse a [`../scripts/download_data.py`](../scripts/download_data.py).

## **2.3 Almacenamiento en la estructura del proyecto**

El script ya cumple con este punto, ya que guarda cada archivo en `data/raw/<tipo>/<anio>/`, que es la estructura definida por el repositorio. Adicional, con la modificación del 2.2 la ruta se resuelve a partir de la ubicación del script y no del directorio desde donde se ejecuta:

```python
DIR_DESTINO = Path(__file__).resolve().parent.parent / "data" / "raw"

def ruta_destino(tipo: str, anio: int, mes: int) -> Path:
    """Ruta local donde se guarda el archivo."""
    return DIR_DESTINO / tipo / str(anio) / construir_nombre(tipo, anio, mes)
```

## **2.4 Evitar descargas repetidas**

El script ya cumple con este punto. Antes de consultar al servidor, se revisa si el archivo existe localmente y, de ser así, se omite:

```python
if destino.exists() and destino.stat().st_size > 0:
    print(f"  {etiqueta}  ya existe, se omite")
    resumen["omitidos"] += 1
    continue
```

Cabe mencionar que esta validación es confiable porque la descarga se hace sobre un archivo temporal `.part`, que solo se renombra al nombre final cuando la descarga terminó y pasó las verificaciones de tamaño y formato. De esta forma, un archivo con el nombre final siempre corresponde a una descarga completa.

## **2.5 Ejecución y verificación de la descarga**

La primera ejecución descargó 16 archivos (enero a agosto de 2026 para taxis amarillos y verdes), sin fallidos; de septiembre a diciembre la TLC aún no los ha publicado. A continuación se ejecuta nuevamente el script, lo cual también evidencia el punto 2.4, ya que los archivos existentes se omiten.

In [1]:
!python ../scripts/download_data.py


=== YELLOW 2026 ===
  2026-01  ya existe, se omite
  2026-02  ya existe, se omite
  2026-03  ya existe, se omite
  2026-04  ya existe, se omite
  2026-05  ya existe, se omite
  2026-06  ya existe, se omite
  2026-07  ya existe, se omite
  2026-08  ya existe, se omite


  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


  2026-11  aun no publicado por la TLC


  2026-12  aun no publicado por la TLC

=== GREEN 2026 ===
  2026-01  ya existe, se omite
  2026-02  ya existe, se omite
  2026-03  ya existe, se omite
  2026-04  ya existe, se omite
  2026-05  ya existe, se omite
  2026-06  ya existe, se omite
  2026-07  ya existe, se omite
  2026-08  ya existe, se omite


  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


  2026-11  aun no publicado por la TLC


  2026-12  aun no publicado por la TLC

RESUMEN
  anios         : 2026
  destino       : /workspace/data/raw
  descargados   : 0
  ya existian   : 16
  no publicados : 8
      yellow 2026-09, yellow 2026-10, yellow 2026-11, yellow 2026-12, green 2026-09, green 2026-10, green 2026-11, green 2026-12
  fallidos      : 0


Luego, se verifica cada mes contra el servidor: para los meses publicados, el archivo local debe existir, tener el mismo tamaño que informa el servidor y poder leerse como Parquet.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, str(Path("../scripts").resolve()))
import download_data as dd

ANIO = 2026
filas = []
for tipo in dd.TIPOS_TAXI:
    for mes in range(1, 13):
        estado, tamanio_servidor, _ = dd.consultar_publicacion(dd.construir_url(tipo, ANIO, mes))
        ruta = dd.ruta_destino(tipo, ANIO, mes)
        existe = ruta.exists()
        filas.append({
            "tipo": tipo,
            "mes": f"{ANIO}-{mes:02d}",
            "estado_servidor": estado,
            "existe_local": existe,
            "tamanio_local": ruta.stat().st_size if existe else None,
            "tamanio_servidor": tamanio_servidor,
            "filas": pq.ParquetFile(ruta).metadata.num_rows if existe else None,
        })

verificacion = pd.DataFrame(filas).astype(
    {"tamanio_local": "Int64", "tamanio_servidor": "Int64", "filas": "Int64"}
)
# La comparacion de tamanio solo aplica a los meses publicados
verificacion["coincide"] = (
    verificacion["tamanio_local"] == verificacion["tamanio_servidor"]
).astype("boolean")
verificacion

,tipo,mes,estado_servidor,existe_local,tamanio_local,tamanio_servidor,filas,coincide
0,yellow,2026-01,publicado,True,64165080,64165080,3724889,True
1,yellow,2026-02,publicado,True,58683353,58683353,3399866,True
2,yellow,2026-03,publicado,True,67891249,67891249,3952451,True
3,yellow,2026-04,publicado,True,64818115,64818115,3831240,True
4,yellow,2026-05,publicado,True,69699174,69699174,4090836,True
5,yellow,2026-06,publicado,True,65465637,65465637,3837248,True
6,yellow,2026-07,publicado,True,61685033,61685033,3530109,True
7,yellow,2026-08,publicado,True,59043961,59043961,3336716,True
8,yellow,2026-09,no_publicado,False,<NA>,<NA>,<NA>,<NA>
9,yellow,2026-10,no_publicado,False,<NA>,<NA>,<NA>,<NA>


In [3]:
publicados = verificacion[verificacion["estado_servidor"] == dd.PUBLICADO]
no_publicados = verificacion[verificacion["estado_servidor"] == dd.NO_PUBLICADO]

assert (verificacion["estado_servidor"] != dd.ERROR).all(), "Hubo errores al consultar el servidor"
assert publicados["existe_local"].all(), "Falta descargar algún mes publicado"
assert publicados["coincide"].all(), "Algún archivo no coincide con el tamaño del servidor"
assert not no_publicados["existe_local"].any(), "Hay archivos locales de meses no publicados"

print(f"[ok] {len(publicados)} archivos publicados, todos descargados y con el tamaño del servidor")
print(f"[ok] {len(no_publicados)} meses aún no publicados por la TLC")
print(f"[ok] {int(publicados['filas'].sum()):,} registros en total")

[ok] 16 archivos publicados, todos descargados y con el tamaño del servidor
[ok] 8 meses aún no publicados por la TLC
[ok] 30,040,469 registros en total


Los archivos descargados se encuentran en [`../data/raw/`](../data/raw/), organizados como `data/raw/<tipo>/<anio>/<tipo>_tripdata_<anio>-<mes>.parquet`.

## **2.6 Cambios realizados al script**

Como fue mencionado en el 2.1 y 2.2, los cambios realizados al script fueron:

- **Años configurables:** se agregó el argumento `--anio`, que acepta uno o varios años (por defecto 2026).
- **Rutas independientes del directorio de ejecución:** el destino se resuelve a partir de la ubicación del script.
- **Manejo de errores:** se distingue entre un mes no publicado (HTTP 403/404) y un error de red o del servidor, el cual se reporta como fallido.
- **Verificación de la descarga:** se valida que el tamaño coincida con el informado por el servidor y que el archivo sea un Parquet legible antes de guardarlo.
- **Espera entre intentos:** los reintentos esperan un tiempo creciente (2 s, 4 s) y se hace una pausa de 1 s entre descargas.

## **2.7 Criterio para determinar que el conjunto de datos está completo**

Para determinar que el conjunto de datos está completo se hicieron dos validaciones. Primero, a nivel de archivos (celdas del 2.5), se consultó al servidor de la TLC qué meses de 2026 están publicados en lugar de asumir los 12, y se comprobó que cada mes publicado existe localmente, que su tamaño coincide exactamente con el informado por el servidor y que se puede leer como Parquet. Con esto se confirma que se descargaron los 16 archivos esperados (enero a agosto, para taxis amarillos y verdes) y que ninguno quedó incompleto.

Luego, a nivel de contenido, se validó con DuckDB que cada archivo tenga registros y que sus viajes correspondan al mes que indica su nombre. No se utilizó únicamente el rango mínimo y máximo de fechas, ya que existen registros con fechas erróneas (por ejemplo, viajes de 2001 o 2008 dentro de archivos de 2026) que harían que el rango no sea representativo; en su lugar, se calculó el porcentaje de viajes que caen fuera del mes del archivo.

In [4]:
import duckdb

cobertura = duckdb.sql(r"""
    WITH viajes AS (
        SELECT regexp_extract(filename, '(yellow|green)_tripdata_(\d{4}-\d{2})', 1) AS tipo,
               regexp_extract(filename, '(yellow|green)_tripdata_(\d{4}-\d{2})', 2) AS mes_archivo,
               strftime(COALESCE(tpep_pickup_datetime, lpep_pickup_datetime), '%Y-%m') AS mes_viaje
        FROM read_parquet('../data/raw/*/2026/*.parquet', filename = true, union_by_name = true)
    )
    SELECT tipo,
           mes_archivo,
           COUNT(*)                                                    AS viajes,
           COUNT(*) FILTER (WHERE mes_viaje = mes_archivo)             AS viajes_en_mes,
           ROUND(100.0 * COUNT(*) FILTER (WHERE mes_viaje <> mes_archivo) / COUNT(*), 4)
                                                                       AS pct_fuera_del_mes
    FROM viajes
    GROUP BY tipo, mes_archivo
    ORDER BY tipo DESC, mes_archivo
""").df()
cobertura

,tipo,mes_archivo,viajes,viajes_en_mes,pct_fuera_del_mes
0,yellow,2026-01,3724889,3724882,0.0002
1,yellow,2026-02,3399866,3399850,0.0005
2,yellow,2026-03,3952451,3952432,0.0005
3,yellow,2026-04,3831240,3831228,0.0003
4,yellow,2026-05,4090836,4090822,0.0003
5,yellow,2026-06,3837248,3837231,0.0004
6,yellow,2026-07,3530109,3530063,0.0013
7,yellow,2026-08,3336716,3336701,0.0004
8,green,2026-01,40272,40250,0.0546
9,green,2026-02,37373,37362,0.0294


In [5]:
meses_esperados = publicados["mes"].unique().tolist()

for tipo in dd.TIPOS_TAXI:
    meses_tipo = cobertura.loc[cobertura["tipo"] == tipo, "mes_archivo"].tolist()
    assert meses_tipo == sorted(meses_esperados), f"{tipo}: faltan meses o hay meses de más"
assert (cobertura["viajes"] > 0).all(), "Hay archivos sin registros"
assert (cobertura["pct_fuera_del_mes"] < 0.1).all(), "Algún archivo tiene muchos viajes fuera de su mes"

print(f"[ok] {len(cobertura)} archivos con registros, de {meses_esperados[0]} a {meses_esperados[-1]} sin huecos")
print(f"[ok] máximo de viajes fuera de su mes: {cobertura['pct_fuera_del_mes'].max()} %")

[ok] 16 archivos con registros, de 2026-01 a 2026-08 sin huecos
[ok] máximo de viajes fuera de su mes: 0.0546 %


Como resultado, ambos tipos de taxi cubren de enero a agosto de 2026 sin meses faltantes, y en todos los archivos más del 99.9 % de los viajes corresponde al mes esperado. Los pocos registros fuera de su mes son problemas de calidad de datos que se analizarán en el Ejercicio 3.